In [42]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# 1. DATA LOADING (Robust path handling)
print("Loading datasets...")
try:
    # Try reading directly if the notebook is in the same folder
    comm = pd.read_csv('communications_data.csv')
    demo = pd.read_csv('demographics_data.csv')
    econ = pd.read_csv('economy_data.csv')
    energy = pd.read_csv('energy_data.csv')
    geog = pd.read_csv('geography_data.csv')
    gov = pd.read_csv('government_and_civics_data.csv')
    trans = pd.read_csv('transportation_data.csv')
except FileNotFoundError:
    # Fallback if running from a parent directory
    comm = pd.read_csv(os.path.join('data', 'communications_data.csv'))
    demo = pd.read_csv(os.path.join('data', 'demographics_data.csv'))
    econ = pd.read_csv(os.path.join('data', 'economy_data.csv'))
    energy = pd.read_csv(os.path.join('data', 'energy_data.csv'))
    geog = pd.read_csv(os.path.join('data', 'geography_data.csv'))
    gov = pd.read_csv(os.path.join('data', 'government_and_civics_data.csv'))
    trans = pd.read_csv(os.path.join('data', 'transportation_data.csv'))

# 2. DATA MERGING
# Merge all datasets starting from demographics (most complete country list)
df = demo.merge(comm, on='Country', how='left') \
         .merge(econ, on='Country', how='left') \
         .merge(energy, on='Country', how='left') \
         .merge(geog, on='Country', how='left') \
         .merge(trans, on='Country', how='left') \
         .merge(gov, on='Country', how='left')

# 3. DATA CLEANING
def clean_numeric(x):
    """Removes symbols like $, %, commas, and units from strings."""
    if isinstance(x, str):
        clean_str = x.replace(',', '').replace('$', '').replace('%', '') \
                     .replace('sq km', '').replace('million', '').strip()
        if not clean_str: return np.nan
        try: return float(clean_str)
        except: return np.nan
    return x

# Apply cleaning to all columns except 'Country'
cols_to_clean = [col for col in df.columns if col != 'Country']
for col in cols_to_clean:
    df[col] = df[col].apply(clean_numeric)

# 4. FEATURE ENGINEERING
# Prevent division by zero
df['Land_Area'] = df['Land_Area'].replace(0, np.nan)
df['Total_Population'] = df['Total_Population'].replace(0, np.nan)

# Basic Densities
df['Road_Density'] = df['roadways_km'] / df['Land_Area']
df['Internet_Penetration'] = df['internet_users_total'] / df['Total_Population']

# Transport Density Metrics (Fixed to accept Series)
def get_density(values_series):
    # Calculate density per 1M population
    per_pop = values_series / df['Total_Population'] * 1_000_000
    # Calculate density per 1000 sq km (fallback)
    per_area = values_series / df['Land_Area'] * 1_000
    # Use per_pop, fill NaNs with per_area
    return per_pop.fillna(per_area)

# Calculate total runways (sum of paved and unpaved)
total_runways = df['airports_paved_runways_count'].fillna(0) + df['airports_unpaved_runways_count'].fillna(0)

# Apply density calculation passing the data Series directly
df['density_runways'] = get_density(total_runways)
df['density_railways'] = get_density(df['railways_km'])
df['density_waterways'] = get_density(df['waterways_km'])
df['density_roads'] = get_density(df['roadways_km'])

# 5. RISK PILLARS CALCULATION
scaler = MinMaxScaler()

def calc_risk(cols, invert_cols=[]):
    """Calculates a normalized risk score (0-1) for a list of columns."""
    temp = df[cols].copy().fillna(df[cols].mean()) # Basic mean imputation

    for c in cols:
        # Handle constant columns
        if temp[c].max() == temp[c].min():
            temp[c] = 0.5
        else:
            vals = temp[[c]].values
            temp[c] = scaler.fit_transform(vals)

        # Invert 'positive' metrics (where High Value = Low Risk)
        if c in invert_cols:
            temp[c] = 1 - temp[c]

    return temp.mean(axis=1)

# Risk 1: Economic Resilience
df['Economic Risk'] = calc_risk(
    ['Population_Below_Poverty_Line_percent', 'Public_Debt_percent_of_GDP', 'Real_GDP_per_Capita_USD'],
    invert_cols=['Real_GDP_per_Capita_USD']
)

# Risk 2: Social Fragility
df['Social Risk'] = calc_risk(
    ['Infant_Mortality_Rate', 'Youth_Unemployment_Rate', 'Net_Migration_Rate'],
    invert_cols=['Net_Migration_Rate']
)

# Risk 3: Infrastructure Access
df['Infrastructure Risk'] = calc_risk(
    ['electricity_access_percent', 'Road_Density', 'Internet_Penetration'],
    invert_cols=['electricity_access_percent', 'Road_Density', 'Internet_Penetration']
)

# Risk 4: Demographic Stress
df['Demographic Risk'] = calc_risk(
    ['Population_Growth_Rate', 'Median_Age'],
    invert_cols=['Median_Age']
)

# Risk 5: Transport Constraint
trans_cols = ['density_runways', 'density_roads', 'density_railways', 'density_waterways']
trans_df = df[trans_cols].copy()
# Impute transport NaNs with median and clip negatives
trans_df = trans_df.fillna(trans_df.median()).clip(lower=0)

# Log -> Norm -> Mean -> Invert
trans_log = trans_df.apply(np.log1p)
trans_norm = pd.DataFrame(scaler.fit_transform(trans_log), columns=trans_cols)
# High Supply = Low Constraint, so we invert
df['Transport Constraint'] = 1 - trans_norm.mean(axis=1)

# 6. FINAL CALCULATION AND PCA
risk_pillars = ['Economic Risk', 'Social Risk', 'Infrastructure Risk', 'Demographic Risk', 'Transport Constraint']
df['Total Vulnerability'] = df[risk_pillars].mean(axis=1)

# PCA Execution
X = df[risk_pillars].values
# Final safety imputation for PCA
X_imputed = SimpleImputer(strategy='mean').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imputed)
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

# Store PCA coordinates
df['PC1'] = components[:, 0]
df['PC2'] = components[:, 1]

# 7. EXPORT DATA
final_cols = ['Country'] + risk_pillars + ['Total Vulnerability', 'PC1', 'PC2']
df[final_cols].to_csv('processed_risk_data.csv', index=False)

print("Processing complete successfully!")
print(f"File saved: processed_risk_data.csv")
print("Columns generated:", df[final_cols].columns.tolist())

Loading datasets...
Processing complete successfully!
File saved: processed_risk_data.csv
Columns generated: ['Country', 'Economic Risk', 'Social Risk', 'Infrastructure Risk', 'Demographic Risk', 'Transport Constraint', 'Total Vulnerability', 'PC1', 'PC2']


In [46]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# --- 1. Explicit Definition of the 5 Pillars ---
# This ensures we are strictly using the 5 expected pillars and not an old 4-pillar list
risk_pillars = [
    'Economic Risk',
    'Social Risk',
    'Infrastructure Risk',
    'Demographic Risk',
    'Transport Constraint'  # Essential: ensure this new pillar is included!
]

print(f"Running PCA analysis on {len(risk_pillars)} pillars: {risk_pillars}")

# --- 2. Recalculate PCA ---
# It is critical to re-run the fit step here to ensure 'Transport Constraint' is included in the model.

# Prepare the data matrix X
X = df[risk_pillars].values

# Data Cleaning (Imputation) and Scaling (Standardization)
# Imputation handles any remaining NaNs; Scaling ensures all risks have equal weight
X_imputed = SimpleImputer(strategy='mean').fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imputed)

# Execute PCA
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

# Update the DataFrame with new PCA coordinates and Total Vulnerability score
df['PC1'] = components[:, 0]
df['PC2'] = components[:, 1]
df['Total Vulnerability'] = df[risk_pillars].mean(axis=1)

# --- 3. Component Analysis (Loadings Matrix) ---
# This section prints the statistics to help interpret what PC1 and PC2 actually represent.

print("\n--- PCA Component Analysis ---")

# Explained Variance Ratio
# Indicates how much of the dataset's total information is captured by the 2D map
explained_var = pca.explained_variance_ratio_
print(f"Explained Variance PC1: {explained_var[0]*100:.2f}%")
print(f"Explained Variance PC2: {explained_var[1]*100:.2f}%")
print(f"Total Explained Variance: {sum(explained_var)*100:.2f}%")

# Loadings Matrix
# Shows the correlation between each Risk Pillar and the Principal Components.
# High absolute values indicate that the risk is a strong driver of that component.
loadings_df = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=risk_pillars
)

print("\nLoadings Matrix (Contribution of each risk to the components):")
print(loadings_df)

# --- 4. Final Data Export ---
# Save the updated dataset to ensure the Dash app reads the correct 5-pillar data.
final_cols = ['Country'] + risk_pillars + ['Total Vulnerability', 'PC1', 'PC2']
output_filename = 'processed_risk_data.csv'
df[final_cols].to_csv(output_filename, index=False)

print(f"\nSuccess! File '{output_filename}' has been updated with 5 risks.")

Running PCA analysis on 5 pillars: ['Economic Risk', 'Social Risk', 'Infrastructure Risk', 'Demographic Risk', 'Transport Constraint']

--- PCA Component Analysis ---
Explained Variance PC1: 55.77%
Explained Variance PC2: 16.59%
Total Explained Variance: 72.36%

Loadings Matrix (Contribution of each risk to the components):
                           PC1       PC2
Economic Risk         0.471781 -0.269370
Social Risk           0.394732 -0.445016
Infrastructure Risk   0.494908  0.111605
Demographic Risk      0.516402 -0.063020
Transport Constraint  0.331670  0.844378

Success! File 'processed_risk_data.csv' has been updated with 5 risks.
